# Add Google Drive links to the Excel files

Run the cell below (Shift+Enter) and allow the two Google pop-ups (mount Drive, then sign in).
Change `DRIVE_FOLDER` inside the cell first if your folder is somewhere else in My Drive.

In [4]:
"""
Make the paths in the Excel files of a find_duplicates.py output folder CLICKABLE
GOOGLE DRIVE LINKS: clicking a path opens that image in Google Drive (in the browser).

It must run in GOOGLE COLAB (colab.research.google.com), because it needs your Google
login to read the Drive file IDs. Steps:
  1. Open https://colab.research.google.com  ->  New notebook.
  2. Upload this file (left panel -> folder icon -> upload) .
  3. In a cell run:      %run add_drive_links.py
  4. Allow the two Google pop-ups (mount Drive, then sign in).

What it does
  * finds the folder DRIVE_FOLDER (below) in your My Drive and reads the ID of every file in it
  * in every .xlsx inside that folder:
      - turns old local paths (F:/...) into relative paths (positive/<dataset>/C1/images/x.jpg)
      - makes the "Path in Output" and "Duplicate Of" cells hyperlinks to
        https://drive.google.com/file/d/<file id>/view
  * originals are first copied to a backup folder next to it: <folder>_excel_backup
Safe to run again.
"""

import os
import re
import shutil
import sys
from collections import deque

# ---------------------------- settings ----------------------------
# Folder path INSIDE "My Drive" that holds the output folder content and the Excel files
DRIVE_FOLDER = "De-duplication stages/polypdataset for class seperation(v4) withoutduplicates"

LINK_HEADER_WORDS = ("path in output", "duplicate of")   # these columns become links
PATH_HEADER_WORDS = ("path", "duplicate of")             # these columns are made relative
CLASS_FOLDERS = {"positive", "negativeonly"}
FOLDER_MIME = "application/vnd.google-apps.folder"
# ------------------------------------------------------------------


# ------------------------- path helpers -------------------------
def looks_absolute(v):
    return bool(re.match(r"^([A-Za-z]:[\\/]|\\\\|/)", v))


def to_relative(value):
    """Old absolute local path -> 'positive/...' ; relative paths are only normalised."""
    if not isinstance(value, str) or not value.strip():
        return value
    norm = value.strip().replace("\\", "/")
    if not looks_absolute(value.strip()):
        return norm
    parts = norm.split("/")
    for i, part in enumerate(parts):
        if part.lower() in CLASS_FOLDERS:
            return "/".join(parts[i:])
    return norm


def drive_url(file_id):
    return f"https://drive.google.com/file/d/{file_id}/view"


# ------------------------- Google Drive API -------------------------
def q_escape(name):
    return name.replace("\\", "\\\\").replace("'", "\\'")


def find_folder_id(service, parts):
    """ID of My Drive/<parts[0]>/<parts[1]>/..."""
    parent = "root"
    for name in parts:
        q = (f"name = '{q_escape(name)}' and '{parent}' in parents "
             f"and mimeType = '{FOLDER_MIME}' and trashed = false")
        found = service.files().list(q=q, fields="files(id, name)", pageSize=10).execute().get("files", [])
        if not found:
            raise SystemExit(f"Folder '{name}' not found in Drive. Check DRIVE_FOLDER at the top of the script.")
        if len(found) > 1:
            print(f"Warning: {len(found)} folders named '{name}' - using the first one.")
        parent = found[0]["id"]
    return parent


def build_id_map(service, root_id):
    """{relative path inside the folder: Drive file id} for every file below root_id."""
    id_map, dupes = {}, []
    queue, folders = deque([(root_id, "")]), 0
    while queue:
        fid, prefix = queue.popleft()
        folders += 1
        token = None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageSize=1000, pageToken=token).execute()
            for f in resp.get("files", []):
                rel = prefix + f["name"]
                if f["mimeType"] == FOLDER_MIME:
                    queue.append((f["id"], rel + "/"))
                elif rel in id_map:
                    dupes.append(rel)
                else:
                    id_map[rel] = f["id"]
            token = resp.get("nextPageToken")
            if not token:
                break
        if folders % 50 == 0:
            print(f"  ...{folders} folders read, {len(id_map)} files found")
    return id_map, dupes


# ------------------------- Excel -------------------------
def apply_links(folder, id_map):
    from openpyxl import load_workbook
    from openpyxl.styles import Font
    from openpyxl.utils import get_column_letter

    link_font = Font(color="0563C1", underline="single")
    backup = folder.rstrip("\\/") + "_excel_backup"
    os.makedirs(backup, exist_ok=True)

    files = sorted(f for f in os.listdir(folder) if f.lower().endswith(".xlsx") and not f.startswith("~$"))
    if not files:
        raise SystemExit(f"No .xlsx files found in {folder}")

    for name in files:
        path = os.path.join(folder, name)
        if not os.path.exists(os.path.join(backup, name)):
            shutil.copy2(path, os.path.join(backup, name))

        wb = load_workbook(path)
        ws = wb.active
        headers = [str(c.value or "").lower() for c in ws[1]]
        cols = [i for i, h in enumerate(headers) if any(w in h for w in PATH_HEADER_WORDS)]
        link_cols = {i for i in cols if any(w in headers[i] for w in LINK_HEADER_WORDS)}
        if not cols:
            print(f"{name}: no path columns, skipped")
            continue

        linked, missing = 0, []
        for row in ws.iter_rows(min_row=2):
            for i in cols:
                cell = row[i]
                new = to_relative(cell.value)
                if new != cell.value:
                    cell.value = new
                if i in link_cols and isinstance(new, str) and new.strip():
                    fid = id_map.get(new.strip())
                    if fid:
                        cell.hyperlink = drive_url(fid)
                        cell.font = link_font
                        linked += 1
                    else:
                        missing.append(new)

        for i in range(1, ws.max_column + 1):
            letter = get_column_letter(i)
            width = max(len(str(c.value)) if c.value is not None else 0 for c in ws[letter])
            ws.column_dimensions[letter].width = min(width + 2, 90)
        wb.save(path)

        msg = f"{name}: {linked} Drive links added"
        if missing:
            msg += f", {len(missing)} paths NOT found in Drive"
        print(msg)
        for m in missing[:5]:
            print(f"    not found: {m}")


# ------------------------- main (Google Colab) -------------------------
def main():
    try:
        from google.colab import auth, drive as colab_drive
    except ImportError:
        sys.exit("This script has to run in Google Colab (colab.research.google.com) - see the top of the file.")

    colab_drive.mount("/content/drive")
    auth.authenticate_user()
    from googleapiclient.discovery import build
    service = build("drive", "v3", cache_discovery=False)

    parts = [p for p in DRIVE_FOLDER.replace("\\", "/").split("/") if p]
    root_id = find_folder_id(service, parts)
    print("Reading the file IDs from Google Drive...")
    id_map, dupes = build_id_map(service, root_id)
    print(f"Found {len(id_map)} files.")
    if dupes:
        print(f"Warning: {len(dupes)} files share a name with another file in the same folder "
              f"(only the first one is linked), e.g. {dupes[0]}")

    folder = os.path.join("/content/drive/MyDrive", *parts)
    apply_links(folder, id_map)
    print(f"\nDone. Original sheets backed up in: {folder.rstrip('/')}_excel_backup")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reading the file IDs from Google Drive...
  ...50 folders read, 4283 files found
  ...100 folders read, 9703 files found
  ...150 folders read, 23867 files found
Found 24563 files.
negativeonly_duplicates.xlsx: 0 Drive links added
negativeonly_originals.xlsx: 4275 Drive links added
positive_duplicates.xlsx: 152 Drive links added
positive_originals.xlsx: 20204 Drive links added

Done. Original sheets backed up in: /content/drive/MyDrive/De-duplication stages/polypdataset for class seperation(v4) withoutduplicates_excel_backup
